In [2]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import math
import copy

import matplotlib.pyplot as plt
from data_loader import *
from utils import *
from VAE import *
from VAE_joint import *
from combined_data_loader import *
from Optimization_multi_node import *
from Optimization_single_node import *
import pickle

In [3]:
class Args:
    def __init__(self):
        # ----- problem / optnet -----
        self.T = 24
        self.base_mva = 100.0
        self.capacity_scale = 4.5
        self.ramp_rate = 0.5
        self.voll = 200.0
        self.vosp = 50.0
        self.M_beta = 1e4
        self.pwl_segments = 10

        # IMPORTANT: add these to match gurobi
        self.reserve_up_ratio = 0.05
        self.reserve_dn_ratio = 0.02
        self.rt_up_ratio = 3.0
        self.rt_dn_ratio = 0.5

        self.N_scen = 20       # <== OptNet真正求解的场景池 (即 K)
        self.S_full = 100       # VAE 现场吐出的大量候选场景数 (S 池)
        self.K_rand = 0       # K里面有多少条纯随机保留(防过拟合)
        self.tau_gumbel = 1.0     # Gumbel Softmax 温度
        self.eps_uniform = 0.1 # 防震荡平滑参数
        self.lambda_div = 1e5   # [新增] 避免多头选到同一个场景的相互排斥惩罚力度

        # ----- 分段训练控制参数 -----
        self.device = "cuda"
       #self.batch_size = 32   # (注意: 原来你的叫 dfl_batch_size, 统一改成 batch_size 给 DataLoader 读)
        self.train_batch_size = 8
        self.test_batch_size = 8
        self.solver = "ECOS"
        
        self.filter_epochs = 5 # Stage 2 (训Filter) 轮数
        self.filter_lr = 1e-3   # Stage 2 学习率
        self.dfl_epochs = 1     # Stage 3 (联合微调) 轮数 (端到端微调极耗时，一般1-3轮即收敛)
        self.dfl_lr = 1e-6      # Stage 3 学习率 (必须极小，防崩坏)
        
args = Args()

### GAN

In [7]:
with open('../Result/GAN/KL/DFL_model_trained_separate_multi_DRO.pkl', 'rb') as f:
    result_separate_multi_DRO_gan = pickle.load(f)

with open('../Result/GAN/compare_res_multi_dro_all_separate.pkl', 'rb') as f:
    compare_res_multi_dro_all_separate_gan = pickle.load(f)

save_run_result(result_separate_multi_DRO_gan, 'separate', out_dir="../Result copy/GAN")
print('=========Separate DRO=========')
for i in compare_res_multi_dro_all_separate_gan['summary_mean'].keys():
    print(i, compare_res_multi_dro_all_separate_gan['summary_mean'][i]['test'])


  Final Results Summary (SEPARATE | MULTI | DRO)
 -> Deterministic Baseline (before DFL training):
    [TEST ] Mean Loss: 398893.625000
 -> Random Baseline (before DFL training):
    [TEST ] Mean Loss: 372536.250000
 -> Random Filter AFTER DFL training:
    [TEST ] Mean Loss: 371985.687500
 -> Fresh ScenarioFilter BEFORE training:
    [TEST ] Mean Loss: 370796.656250
 -> ScenarioFilter AFTER training:
    [TEST ] Mean Loss: 371051.531250

Saved at: ../Result copy/GAN/DFL_model_trained_separate_multi_DRO.pkl
=========Separate DRO=========
random 371985.71875
kmeans 372014.3125
kmedoids 374184.78125
hierarchical 371960.46875
learned (KL) 371051.5
learned (Inner) 371537.03125
learned (Entropy) 371022.625


In [8]:
with open('../Result/GAN/KL/DFL_model_trained_joint_multi_DRO.pkl', 'rb') as f:
    result_joint_multi_DRO_gan = pickle.load(f)

with open('../Result/GAN/compare_res_multi_dro_all_joint.pkl', 'rb') as f:
    compare_res_multi_dro_all_joint_gan = pickle.load(f)

save_run_result(result_joint_multi_DRO_gan, 'joint', out_dir="../Result copy/GAN")
print('=========Joint DRO=========')
for i in compare_res_multi_dro_all_joint_gan['summary_mean'].keys():
    print(i, compare_res_multi_dro_all_joint_gan['summary_mean'][i]['test'])


  Final Results Summary (JOINT | MULTI | DRO)
 -> Random Baseline (before DFL training):
    [TEST ] Mean Loss: 354609.906250
 -> Random Filter AFTER DFL training:
    [TEST ] Mean Loss: 354189.781250
 -> Fresh ScenarioFilter BEFORE training:
    [TEST ] Mean Loss: 352405.593750
 -> ScenarioFilter AFTER training:
    [TEST ] Mean Loss: 352048.218750

Saved at: ../Result copy/GAN/DFL_model_trained_joint_multi_DRO.pkl
=========Joint DRO=========
random 354189.8125
kmeans 354015.9375
kmedoids 360248.96875
hierarchical 353812.0
learned (KL) 352048.1875
learned (Inner) 352333.9375
learned (Entropy) 351980.78125


### Diffusion

In [18]:
with open('../Result/Diffusion/KL/DFL_model_trained_joint_multi_DRO.pkl', 'rb') as f:
    result_joint_multi_DRO_diffusion = pickle.load(f)

with open('../Result/Diffusion/compare_res_multi_dro_all_joint.pkl', 'rb') as f:
    compare_res_multi_dro_all_joint_diffusion = pickle.load(f)

save_run_result(result_joint_multi_DRO_diffusion, 'joint', out_dir="../Result copy/Diffusion")
print('=========Joint DRO=========')
for i in compare_res_multi_dro_all_joint_diffusion['summary_mean'].keys():
    print(i, compare_res_multi_dro_all_joint_diffusion['summary_mean'][i]['test'])


  Final Results Summary (JOINT | MULTI | DRO)
 -> Random Baseline (before DFL training):
    [TEST ] Mean Loss: 366143.125000
 -> Random Filter AFTER DFL training:
    [TEST ] Mean Loss: 362283.250000
 -> Fresh ScenarioFilter BEFORE training:
    [TEST ] Mean Loss: 357032.031250
 -> ScenarioFilter AFTER training:
    [TEST ] Mean Loss: 356104.781250

Saved at: ../Result copy/Diffusion/DFL_model_trained_joint_multi_DRO.pkl
=========Joint DRO=========
random 362283.21875
kmeans 360491.46875
kmedoids 369703.65625
hierarchical 360290.4375
learned (KL) 356104.78125
learned (Inner) 356480.75
learned (Entropy) 358064.5625


In [17]:
with open('../Result/Diffusion/KL/DFL_model_trained_separate_multi_DRO.pkl', 'rb') as f:
    result_separate_multi_DRO_diffusion = pickle.load(f)

with open('../Result/Diffusion/compare_res_multi_dro_all_separate.pkl', 'rb') as f:
    compare_res_multi_dro_all_separate_diffusion = pickle.load(f)

save_run_result(result_separate_multi_DRO_diffusion, 'separate', out_dir="../Result copy/Diffusion")
print('=========Separate DRO=========')
for i in compare_res_multi_dro_all_separate_diffusion['summary_mean'].keys():
    print(i, compare_res_multi_dro_all_separate_diffusion['summary_mean'][i]['test'])


  Final Results Summary (SEPARATE | MULTI | DRO)
 -> Deterministic Baseline (before DFL training):
    [TEST ] Mean Loss: 411669.500000
 -> Random Baseline (before DFL training):
    [TEST ] Mean Loss: 367915.937500
 -> Random Filter AFTER DFL training:
    [TEST ] Mean Loss: 367544.093750
 -> Fresh ScenarioFilter BEFORE training:
    [TEST ] Mean Loss: 366977.218750
 -> ScenarioFilter AFTER training:
    [TEST ] Mean Loss: 366599.562500

Saved at: ../Result copy/Diffusion/DFL_model_trained_separate_multi_DRO.pkl
=========Separate DRO=========
random 367544.09375
kmeans 367216.15625
kmedoids 370047.3125
hierarchical 367179.03125
learned (KL) 366599.5625
learned (Inner) 366991.03125
learned (Entropy) 366745.84375


### VAE

In [20]:
result_joint_multi_DRO_vae.keys()

dict_keys(['optimization_mode', 'problem_mode', 'eval_splits', 'eval_flags', 'run_stage2', 'run_stage3', 'reused_stage2', 'dfl_det_before', 'dfl_before', 'dfl_trained', 'train_logs_stage2', 'train_logs_stage3', 'time_stage2_sec', 'time_stage3_sec', 'train_time_sec_total', 'train_batch_size_used', 'test_batch_size_used', 'N_scen', 'S_full', 'K_rand', 'eval_mode', 'avoid_rand_duplicate', 'stage2_artifact', 'test_losses_stage1_after', 'test_losses_stage2_after', 'test_losses_stage3_before', 'test_losses_stage3_after', 'test_losses_random_baseline', 'test_losses_random_filter_after_dfl', 'test_losses_scenario_filter_before_training', 'test_losses_scenario_filter_after_training', 'test_losses_before_filter_training', 'test_losses_after_filter_training', 'test_losses_before', 'test_losses_after'])

In [12]:
with open('../Result/VAE/KL/DFL_model_trained_joint_multi_DRO.pkl', 'rb') as f:
    result_joint_multi_DRO_vae = pickle.load(f)

with open('../Result/VAE/compare_res_multi_dro_all_joint.pkl', 'rb') as f:
    compare_res_multi_dro_all_joint_vae = pickle.load(f)

save_run_result(result_joint_multi_DRO_vae, 'joint', out_dir="../Result copy/VAE")
print('=========Joint DRO=========')
for i in compare_res_multi_dro_all_joint_vae['summary_mean'].keys():
    print(i, compare_res_multi_dro_all_joint_vae['summary_mean'][i]['test'])


  Final Results Summary (JOINT | MULTI | DRO)
 -> Random Baseline (before DFL training):
    [TEST ] Mean Loss: 355684.781250
 -> Random Filter AFTER DFL training:
    [TEST ] Mean Loss: 351629.187500
 -> Fresh ScenarioFilter BEFORE training:
    [TEST ] Mean Loss: 352654.437500
 -> ScenarioFilter AFTER training:
    [TEST ] Mean Loss: 349299.500000

Saved at: ../Result copy/VAE/DFL_model_trained_joint_multi_DRO.pkl
=========Joint DRO=========
random 351629.15625
kmeans 350304.59375
kmedoids 352800.21875
hierarchical 350236.96875
learned (KL) 349299.5
learned (Inner) 349609.96875
learned (Entropy) 349589.5625


In [23]:
with open('../Result/VAE/KL/DFL_model_trained_separate_multi_DRO.pkl', 'rb') as f:
    result_separate_multi_DRO_vae = pickle.load(f)

with open('../Result/VAE/compare_res_multi_dro_all_separate.pkl', 'rb') as f:
    compare_res_multi_dro_all_separate_vae = pickle.load(f)

save_run_result(result_separate_multi_DRO_vae, 'separate', out_dir="../Result copy/VAE")
print('=========Separate DRO=========')
for i in compare_res_multi_dro_all_separate_vae['summary_mean'].keys():
    print(i, compare_res_multi_dro_all_separate_vae['summary_mean'][i]['test'])


  Final Results Summary (SEPARATE | MULTI | DRO)
 -> Random Baseline (before DFL training):
    [TEST ] Mean Loss: 374705.562500
 -> Random Filter AFTER DFL training:
    [TEST ] Mean Loss: 372933.968750
 -> Fresh ScenarioFilter BEFORE training:
    [TEST ] Mean Loss: 371409.656250
 -> ScenarioFilter AFTER training:
    [TEST ] Mean Loss: 371435.562500

Saved at: ../Result copy/VAE/DFL_model_trained_separate_multi_DRO.pkl
=========Separate DRO=========
random 372934.0
kmeans 372293.8125
kmedoids 374837.53125
hierarchical 372222.59375
learned (KL) 370978.25
learned (Inner) 371471.40625
learned (Entropy) 371783.4375


### Non_parametric

In [13]:
with open('../Result/Non_parametric/KL/DFL_model_trained_separate_multi_DRO.pkl', 'rb') as f:
    result_separate_multi_DRO_non_parametric = pickle.load(f)

with open('../Result/Non_parametric/compare_res_multi_dro_all.pkl', 'rb') as f:
    compare_res_multi_dro_all_separate_non_parametric = pickle.load(f)


In [14]:
save_run_result(result_separate_multi_DRO_non_parametric, 'separate', out_dir="../Result copy/NonParametric/KL/")

print('=========Separate DRO=========')
for i in compare_res_multi_dro_all_separate_non_parametric['summary_mean'].keys():
    print(i, compare_res_multi_dro_all_separate_non_parametric['summary_mean'][i]['test'])


  Final Results Summary (SEPARATE | MULTI | DRO)
 -> Deterministic Baseline (before DFL training):
    [TEST ] Mean Loss: 425584.031250
 -> Random Baseline (before DFL training):
    [TEST ] Mean Loss: 379875.843750
 -> Random Filter AFTER DFL training:
    [TEST ] Mean Loss: 379314.218750
 -> Fresh ScenarioFilter BEFORE training:
    [TEST ] Mean Loss: 379028.000000
 -> ScenarioFilter AFTER training:
    [TEST ] Mean Loss: 379047.187500

Saved at: ../Result copy/NonParametric/KL/DFL_model_trained_separate_multi_DRO.pkl
=========Separate DRO=========
random 379314.21875
kmeans 380288.6875
kmedoids 381859.65625
hierarchical 380528.46875
learned (KL) 379047.1875
learned (Inner) 379018.96875
learned (Entropy) 379004.0


### Parametric

In [15]:
with open('../Result/Parametric/KL/DFL_model_trained_separate_multi_DRO.pkl', 'rb') as f:
    result_separate_multi_DRO_parametric = pickle.load(f)

with open('../Result/Parametric/compare_res_multi_dro_all.pkl', 'rb') as f:
    compare_res_multi_dro_all_separate_parametric = pickle.load(f)

In [16]:
save_run_result(result_separate_multi_DRO_parametric, 'separate', out_dir="../Result copy/Parametric/KL/")

print('=========Separate DRO=========')
for i in compare_res_multi_dro_all_separate_parametric['summary_mean'].keys():
    print(i, compare_res_multi_dro_all_separate_parametric['summary_mean'][i]['test'])


  Final Results Summary (SEPARATE | MULTI | DRO)
 -> Random Baseline (before DFL training):
    [TEST ] Mean Loss: 379128.062500
 -> Random Filter AFTER DFL training:
    [TEST ] Mean Loss: 379176.937500
 -> Fresh ScenarioFilter BEFORE training:
    [TEST ] Mean Loss: 378894.937500
 -> ScenarioFilter AFTER training:
    [TEST ] Mean Loss: 378735.656250

Saved at: ../Result copy/Parametric/KL/DFL_model_trained_separate_multi_DRO.pkl
=========Separate DRO=========
random 379176.9375
kmeans 380402.78125
kmedoids 382379.8125
hierarchical 380578.8125
learned (KL) 378735.65625
learned (Inner) 378795.15625
learned (Entropy) 378796.0


In [19]:
import pickle
import torch
import pandas as pd

def load_pkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

def extract_test_column(compare_res, split='test', block='summary_mean'):
    d = compare_res[block]
    #if type(d[list(d.keys())[0]]) == dict:
    return pd.Series({k: d[k][split] for k in d.keys()})
    #else:
    #    return pd.Series({k: d[k] for k in d.keys()})

def calc_AO_CO_from_result(result_obj):
    # AO = after filter training (stage2 after)
    ao = torch.mean(result_obj['test_losses_stage1_after']).item()
    # CO = random/before filter training
    co = torch.mean(result_obj['test_losses_stage2_after']).item()
    return ao, co

cols = {}

compare_diff_separate = load_pkl('../Result/Diffusion/compare_res_multi_dro_all_separate.pkl')
cols['Diffusion(Separate)'] = extract_test_column(compare_diff_separate)

compare_diff_joint = load_pkl('../Result/Diffusion/compare_res_multi_dro_all_joint.pkl')
cols['Diffusion(Joint)'] = extract_test_column(compare_diff_joint)

compare_gan_joint = load_pkl('../Result/GAN/compare_res_multi_dro_all_joint.pkl')
cols['GAN(Joint)'] = extract_test_column(compare_gan_joint)

compare_gan_separate = load_pkl('../Result/GAN/compare_res_multi_dro_all_separate.pkl')
cols['GAN(Separate)'] = extract_test_column(compare_gan_separate)

compare_vae_sep = load_pkl('../Result/VAE/compare_res_multi_dro_all_separate.pkl')
cols['VAE(Separate)'] = extract_test_column(compare_vae_sep)

compare_vae_joint = load_pkl('../Result/VAE/compare_res_multi_dro_all_joint.pkl')
cols['VAE(Joint)'] = extract_test_column(compare_vae_joint)

compare_np = load_pkl('../Result/Non_parametric/compare_res_multi_dro_all.pkl')
cols['Non-parametric'] = extract_test_column(compare_np)

compare_p = load_pkl('../Result/Parametric/compare_res_multi_dro_all.pkl')
cols['Parametric'] = extract_test_column(compare_p)

df = pd.DataFrame(cols)

row_order = list(compare_vae_sep['summary_mean'].keys())
df = df.reindex(row_order)

# --------- 2) AO/CO 两列（每个方法各自一行的 AO/CO，放在df左侧）---------
# 读各方法的 result pkl
result_diff_separate = load_pkl('../Result/Diffusion/KL/DFL_model_trained_separate_multi_DRO.pkl')
result_diff_joint = load_pkl('../Result/Diffusion/KL/DFL_model_trained_joint_multi_DRO.pkl')
result_gan_joint = load_pkl('../Result/GAN/KL/DFL_model_trained_joint_multi_DRO.pkl')
result_gan_separate = load_pkl('../Result/GAN/KL/DFL_model_trained_separate_multi_DRO.pkl')
result_vae_sep   = load_pkl('../Result/VAE/KL/DFL_model_trained_separate_multi_DRO.pkl')
result_vae_joint = load_pkl('../Result/VAE/KL/DFL_model_trained_joint_multi_DRO.pkl')
result_np        = load_pkl('../Result/Non_parametric/KL/DFL_model_trained_separate_multi_DRO.pkl')
result_p         = load_pkl('../Result/Parametric/KL/DFL_model_trained_separate_multi_DRO.pkl')

ao_co = {
    'Parametric':      calc_AO_CO_from_result(result_p),
    'Non-parametric':  calc_AO_CO_from_result(result_np),
    'VAE(Separate)':   calc_AO_CO_from_result(result_vae_sep),
    'VAE(Joint)':      calc_AO_CO_from_result(result_vae_joint),
    'GAN(Separate)':   calc_AO_CO_from_result(result_gan_separate),
    'GAN(Joint)':      calc_AO_CO_from_result(result_gan_joint),
    'Diffusion(Separate)':   calc_AO_CO_from_result(result_diff_separate),
    'Diffusion(Joint)':calc_AO_CO_from_result(result_diff_joint),
}

ao_co_df = pd.DataFrame(
    {k: {'AO': v[0], 'CO': v[1]} for k, v in ao_co.items()}
).T  # index=方法，columns=['AO','CO']

# 你当前 df 是 index=指标、columns=方法
# 想把 AO/CO 作为“第一列、第二列”放进同一张表，
# 需要先把 df 转置成 index=方法，再拼 AO/CO，再转回来（取决于你想要的最终形状）

df_method_major = df.T  # index=方法, columns=指标
df_method_major = pd.concat([ao_co_df[['AO','CO']], df_method_major], axis=1)

# 如果你最终要和之前一样（行=指标，列=方法），就再转回去：
final_df = df_method_major.T
final_df = final_df.drop(index='random', errors='ignore')

print(final_df)

                      Parametric  Non-parametric  VAE(Separate)  \
AO                 379128.065926   379875.859255  374705.551451   
CO                 379176.952596   379314.216273  372933.979348   
kmeans             380402.781250   380288.687500  372293.812500   
kmedoids           382379.812500   381859.656250  374837.531250   
hierarchical       380578.812500   380528.468750  372222.593750   
learned (KL)       378735.656250   379047.187500  370978.250000   
learned (Inner)    378795.156250   379018.968750  371471.406250   
learned (Entropy)  378796.000000   379004.000000  371783.437500   

                      VAE(Joint)  GAN(Separate)     GAN(Joint)  \
AO                 355684.780893  372536.219652  354609.882536   
CO                 351629.158642  371985.703044  354189.789694   
kmeans             350304.593750  372014.312500  354015.937500   
kmedoids           352800.218750  374184.781250  360248.968750   
hierarchical       350236.968750  371960.468750  353812.000000   


In [8]:
final_df.to_csv('../Result/cost_result.csv')